In [0]:
%pip install geopandas shapely

In [0]:
%restart_python

In [0]:
import geopandas as gpd
import pandas as pd
from shapely import wkt

PROJECTED_CRS = "EPSG:3310"
WGS84_CRS = "EPSG:4326"

SILVER_ZONES_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/analysis_zones"
)

SILVER_HAZARD_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/fire_hazard"
)

SILVER_FIRE_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/fire_history"
)

SILVER_STATIC_PATH = (
    "abfss://wildfire-data@stscewildfirevr01.dfs.core.windows.net/"
    "silver/zone_static_features"
)

print("Configuration loaded.")

In [0]:
zones_pdf = (
    spark.read.format("delta")
    .load(SILVER_ZONES_PATH)
    .toPandas()
)

hazard_pdf = (
    spark.read.format("delta")
    .load(SILVER_HAZARD_PATH)
    .toPandas()
)

fires_pdf = (
    spark.read.format("delta")
    .load(SILVER_FIRE_PATH)
    .toPandas()
)

In [0]:
zones_pdf["geometry"] = zones_pdf["geometry_wkt"].apply(wkt.loads)

zones_final = gpd.GeoDataFrame(
    zones_pdf.drop(columns=["geometry_wkt"]),
    geometry="geometry",
    crs=PROJECTED_CRS
)

hazard_pdf["geometry"] = hazard_pdf["geometry_wkt"].apply(wkt.loads)

hazard_gdf = gpd.GeoDataFrame(
    hazard_pdf.drop(columns=["geometry_wkt"]),
    geometry="geometry",
    crs=WGS84_CRS
)

hazard_projected = hazard_gdf.to_crs(PROJECTED_CRS)

fires_pdf["geometry"] = fires_pdf["geometry_wkt"].apply(wkt.loads)

fires_gdf = gpd.GeoDataFrame(
    fires_pdf.drop(columns=["geometry_wkt"]),
    geometry="geometry",
    crs=WGS84_CRS
)

fires_projected = fires_gdf.to_crs(PROJECTED_CRS)

print("Zones:", len(zones_final))
print("Hazard polygons:", len(hazard_projected))
print("Historical fires:", len(fires_projected))

In [0]:
zone_fire = gpd.overlay(
    zones_final[
        ["zone_id", "geometry"]
    ],
    fires_projected[
        [
            "fire_id",
            "fire_year",
            "fire_name",
            "geometry"
        ]
    ],
    how="intersection",
    keep_geom_type=True
)

print(
    "Zone-fire intersections:",
    len(zone_fire)
)

In [0]:
zone_fire["burned_acres"] = (
    zone_fire.geometry.area
    / 4046.8564224
)

fire_features = (
    zone_fire
    .groupby("zone_id")
    .agg(
        historical_fire_count=(
            "fire_id",
            "nunique"
        ),
        historical_burned_acres=(
            "burned_acres",
            "sum"
        ),
        last_fire_year=(
            "fire_year",
            "max"
        )
    )
    .reset_index()
)

print(
    "Zones with historical fire:",
    len(fire_features)
)

In [0]:
zone_hazard = gpd.overlay(
    zones_final[
        ["zone_id", "geometry"]
    ],
    hazard_projected[
        [
            "hazard_class",
            "hazard_score",
            "geometry"
        ]
    ],
    how="intersection",
    keep_geom_type=True
)

zone_hazard["overlap_area"] = (
    zone_hazard.geometry.area
)

zone_areas = zones_final[
    ["zone_id", "geometry"]
].copy()

zone_areas["zone_area"] = (
    zone_areas.geometry.area
)

zone_areas = zone_areas[
    ["zone_id", "zone_area"]
]

zone_hazard = zone_hazard.merge(
    zone_areas,
    on="zone_id",
    how="left"
)

zone_hazard["area_fraction"] = (
    zone_hazard["overlap_area"]
    /
    zone_hazard["zone_area"]
)

zone_hazard["hazard_contribution"] = (
    zone_hazard["hazard_score"]
    *
    zone_hazard["area_fraction"]
)

zone_hazard_scores = (
    zone_hazard
    .groupby(
        "zone_id",
        as_index=False
    )["hazard_contribution"]
    .sum()
    .rename(
        columns={
            "hazard_contribution":
            "hazard_score"
        }
    )
)

zone_hazard_features = (
    zones_final[
        ["zone_id", "geometry"]
    ]
    .merge(
        zone_hazard_scores,
        on="zone_id",
        how="left"
    )
)

zone_hazard_features["hazard_score"] = (
    zone_hazard_features["hazard_score"]
    .fillna(0)
    .clip(0, 3)
    .round(3)
)

print(
    "Zones scored:",
    len(zone_hazard_features)
)

print(
    "Hazard range:",
    zone_hazard_features["hazard_score"].min(),
    "to",
    zone_hazard_features["hazard_score"].max()
)

In [0]:
zone_features = (
    zone_hazard_features[
        [
            "zone_id",
            "hazard_score",
            "geometry"
        ]
    ]
    .merge(
        fire_features,
        on="zone_id",
        how="left"
    )
)

zone_features["historical_fire_count"] = (
    zone_features["historical_fire_count"]
    .fillna(0)
    .astype(int)
)

zone_features["historical_burned_acres"] = (
    zone_features["historical_burned_acres"]
    .fillna(0)
)

In [0]:
print("Rows:", len(zone_features))

print(
    "Unique zones:",
    zone_features["zone_id"].nunique()
)

print(
    "Hazard range:",
    zone_features["hazard_score"].min(),
    "to",
    zone_features["hazard_score"].max()
)

print(
    "Zones with no recorded fires:",
    (
        zone_features[
            "historical_fire_count"
        ] == 0
    ).sum()
)

print(
    "Historical fire year range:",
    zone_features[
        "last_fire_year"
    ].min(),
    "to",
    zone_features[
        "last_fire_year"
    ].max()
)

In [0]:
static_features = zone_features[
    [
        "zone_id",
        "hazard_score",
        "historical_fire_count",
        "historical_burned_acres",
        "last_fire_year",
        "geometry"
    ]
].copy()

static_features["historical_burned_acres"] = (
    static_features["historical_burned_acres"]
    .round(3)
)

static_features["geometry_wkt"] = (
    static_features.geometry.apply(
        lambda geom: geom.wkt
        if geom is not None
        else None
    )
)

static_features = static_features.drop(
    columns=["geometry"]
)

print("Rows prepared:", len(static_features))
print(static_features.head())

In [0]:
static_spark_df = spark.createDataFrame(
    static_features
)

(
    static_spark_df.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_STATIC_PATH)
)

print("Static zone features written successfully")
print("Path:", SILVER_STATIC_PATH)

In [0]:
check_static = (
    spark.read
    .format("delta")
    .load(SILVER_STATIC_PATH)
)

print("Rows:", check_static.count())

print(
    "Unique zones:",
    check_static
    .select("zone_id")
    .distinct()
    .count()
)

display(
    check_static
    .select(
        "zone_id",
        "hazard_score",
        "historical_fire_count",
        "historical_burned_acres",
        "last_fire_year"
    )
    .limit(10)
)